# 16 — Paper Figures: Classifier and Augmentation Choice

Three panels. (a) and (b) are ranked bar-and-dot comparisons in the
SEP_DataAugmentation-2 style. (c) is new: it answers *why* classical
oversampling (SMOTE/ADASYN) beats generative augmentation (TimeGAN/
Diffusion) rather than just reporting that it does.

The mechanism is not what a first guess would suggest. False-alarm rate
(FAR) is nearly identical across all four algorithms (~0.95-0.96) --
generative methods are not more cautious. The entire gap is **recall**:
classical methods catch ~55% of real SEP events, generative methods
catch only ~36-41%. Panel (c) makes that the headline instead of a
footnote.

**Reads:** `./results/*.txt`. **Requires:** notebook 14 run first.
**Writes:** `./paper/figures/design_choices.pdf` / `.png`.


## 1. Load and Preview

In [3]:
# ══════════════════════════════════════════════════════════════
# Best configuration per classifier (same selection notebook 12 uses:
# highest mean TSS per classifier across the whole pipeline).
# ══════════════════════════════════════════════════════════════

BEST_CONFIG = {
    "GRU": "hybrid_tomek",
    "PatchTST": "tomek_rus8000_adasyn8000",
    "SVM": "minmax_all",
    "InceptionTime": "tomek_nonsep_8000",
}
CLF_ORDER = sorted(BEST_CONFIG, key=lambda c: -load_col(
    [k for k, v in CLASSIFIER_LABELS.items() if v == c][0], BEST_CONFIG[c]).mean())

# Augmentation algorithms at their 500-size variant, pooled across all
# four classifiers (matches notebook 12's cross-classifier comparison).
AUG_ORDER = ["ADASYN", "SMOTE", "TimeGAN", "Diffusion"]
AUG_KEYS  = {"ADASYN": "tomek_rus500_adasyn500", "SMOTE": "tomek_rus500_smote500",
            "TimeGAN": "timegan_500", "Diffusion": "diffusion_500"}
AUG_FAMILY = {"ADASYN": "classical", "SMOTE": "classical",
             "TimeGAN": "generative", "Diffusion": "generative"}
FAMILY_COLOR = {"classical": TEAL, "generative": VERM}

print("Best config per classifier:")
for clf in CLF_ORDER:
    key_slug = {v: k for k, v in CLASSIFIER_LABELS.items()}[clf]
    v = load_col(key_slug, BEST_CONFIG[clf])
    print(f"  {clf:<14} {BEST_CONFIG[clf]:<26} mean TSS={v.mean():.3f}")

print("\nAugmentation algorithm comparison (pooled, n=8 each):")
for a in AUG_ORDER:
    v = load_all_classifiers(AUG_KEYS[a])
    r = load_all_classifiers(AUG_KEYS[a], col=COLUMN["recall"])
    f = load_all_classifiers(AUG_KEYS[a], col=COLUMN["far"])
    print(f"  {a:<10} ({AUG_FAMILY[a]:<10}) TSS={v.mean():.3f}  "
          f"Recall={r.mean():.3f}  FAR={f.mean():.3f}")


Best config per classifier:
  GRU            hybrid_tomek               mean TSS=0.671
  PatchTST       tomek_rus8000_adasyn8000   mean TSS=0.623
  SVM            minmax_all                 mean TSS=0.505
  InceptionTime  tomek_nonsep_8000          mean TSS=0.404

Augmentation algorithm comparison (pooled, n=8 each):
  ADASYN     (classical ) TSS=0.452  Recall=0.555  FAR=0.952
  SMOTE      (classical ) TSS=0.449  Recall=0.559  FAR=0.955
  TimeGAN    (generative) TSS=0.275  Recall=0.360  FAR=0.961
  Diffusion  (generative) TSS=0.323  Recall=0.412  FAR=0.955


## Figure — Base Learner, Augmentation, and Why It Works

In [4]:
# FIG -- design choices: base learner, augmentation, and mechanism
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.35))

# ---------------------------------------------------------------
# (a) Base learner -- best pipeline per classifier
# ---------------------------------------------------------------
ax = axes[0]
for i, clf in enumerate(CLF_ORDER):
    key_slug = {v: k for k, v in CLASSIFIER_LABELS.items()}[clf]
    v = load_col(key_slug, BEST_CONFIG[clf])
    col = TEAL if i == 0 else "0.62"
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    ax.scatter(np.full_like(v, i, dtype=float), v, s=5, color="0.15", lw=0, zorder=5)
    ax.text(i, v.max() + 0.045, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.set_xticks(range(len(CLF_ORDER)))
ax.set_xticklabels(CLF_ORDER, rotation=32, ha="right", fontsize=5.9)
ax.set_ylim(0, 0.85)
finish(ax, ylab="Test TSS")
ax.set_title("(a)  Base learner", loc="left", fontsize=7.0, pad=4)

# ---------------------------------------------------------------
# (b) Augmentation algorithm -- classical vs generative, colored by family
# ---------------------------------------------------------------
ax = axes[1]
for i, a in enumerate(AUG_ORDER):
    v = load_all_classifiers(AUG_KEYS[a])
    col = FAMILY_COLOR[AUG_FAMILY[a]]
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    jitter = np.linspace(-0.09, 0.09, len(v))
    ax.scatter(np.full_like(v, i, dtype=float) + jitter, v, s=5, color="0.15",
              lw=0, alpha=0.75, zorder=5)
    ax.text(i, v.max() + 0.045, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.set_xticks(range(len(AUG_ORDER)))
ax.set_xticklabels(AUG_ORDER, rotation=32, ha="right", fontsize=5.9)
for tick, a in zip(ax.get_xticklabels(), AUG_ORDER):
    tick.set_color(FAMILY_COLOR[AUG_FAMILY[a]])
ax.set_ylim(0, 0.78)
finish(ax, ylab="Test TSS")
ax.set_title("(b)  Augmentation", loc="left", fontsize=7.0, pad=4)
ax.text(0.02, 0.99, "classical", transform=ax.transAxes, fontsize=5.8,
        color=TEAL, ha="left", va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.text(0.98, 0.99, "generative", transform=ax.transAxes, fontsize=5.8,
        color=VERM, ha="right", va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))

# ---------------------------------------------------------------
# (c) Mechanism -- recall (POD) by algorithm; FAR is flat across all four,
# annotated as a small note rather than a bar, since the finding here is
# that FAR does NOT vary while recall does.
# ---------------------------------------------------------------
ax = axes[2]
far_vals = []
for i, a in enumerate(AUG_ORDER):
    v = load_all_classifiers(AUG_KEYS[a], col=COLUMN["recall"])
    far_vals.append(load_all_classifiers(AUG_KEYS[a], col=COLUMN["far"]).mean())
    col = FAMILY_COLOR[AUG_FAMILY[a]]
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    jitter = np.linspace(-0.09, 0.09, len(v))
    ax.scatter(np.full_like(v, i, dtype=float) + jitter, v, s=5, color="0.15",
              lw=0, alpha=0.75, zorder=5)
    ax.text(i, v.max() + 0.05, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.set_xticks(range(len(AUG_ORDER)))
ax.set_xticklabels(AUG_ORDER, rotation=32, ha="right", fontsize=5.9)
for tick, a in zip(ax.get_xticklabels(), AUG_ORDER):
    tick.set_color(FAMILY_COLOR[AUG_FAMILY[a]])
ax.set_ylim(0, 0.95)
finish(ax, ylab="Recall (POD)")
ax.set_title("(c)  Why: detection, not caution", loc="left", fontsize=7.0, pad=4)
# FAR range (0.95-0.96 for all four) is already stated in the figure caption --
# no in-plot annotation needed here, which avoids fighting the dot cloud for space.

fig.subplots_adjust(left=0.075, right=0.99, top=0.90, bottom=0.27, wspace=0.42)
fig.savefig(f"{FIG_DIR}/design_choices.pdf")
fig.savefig(f"{FIG_DIR}/design_choices.png", dpi=340)
plt.show()
print("Saved design_choices.pdf/png")
print(f"FAR range across all four algorithms: {min(far_vals):.3f}-{max(far_vals):.3f}")


Saved design_choices.pdf/png
FAR range across all four algorithms: 0.952-0.961


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_41137/2230029978.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
